---

Reconecte ao Drive.

In [1]:
import os
from pathlib import Path

DRIVE_DIRECTORY = Path.cwd().parent / "data"

---

In [2]:
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## K-means

### Base de dados cobertura vegetal

Neste exercício você vai usar a base de dados de cobertura vegetal, explorada nos exercícios da **Parte 1 - Classificação**.

Comece carregando os dados do arquivo `cov_types.csv` armazenado na pasta do Drive.

In [4]:
base = pd.read_csv(os.path.join(DRIVE_DIRECTORY, 'cov_types.csv'))

Exiba os dados para uma inspeção inicial.

In [5]:
base.head()

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type,Cover_Type
0,2767.0,66.0,17.0,210.0,18.0,1190.0,234.0,204.0,96.0,2251.0,2,30,Lodgepole Pine
1,2724.0,160.0,19.0,60.0,4.0,1350.0,236.0,240.0,127.0,2514.0,2,16,Lodgepole Pine
2,2360.0,65.0,7.0,127.0,21.0,1377.0,227.0,226.0,134.0,339.0,3,5,Ponderosa Pine
3,2995.0,45.0,4.0,285.0,30.0,5125.0,221.0,231.0,146.0,5706.0,0,11,Lodgepole Pine
4,2400.0,106.0,27.0,150.0,63.0,342.0,253.0,196.0,51.0,811.0,2,3,Ponderosa Pine


Desta vez, nós vamos remover os atributos categóricos, `Wilderness_Area` e `Soil_Type`, pois para apresentá-los ao algoritmo, nós precisaríamos usar One Hot Encoding, o que adiciona muitas colunas no dataset, e como nossos dados são limitados a 10 mil exemplos, isso pode prejudicar o algoritmo K-Means.

In [6]:
base = base.drop(["Wilderness_Area", "Soil_Type"], axis=1)
base

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Cover_Type
0,2767.0,66.0,17.0,210.0,18.0,1190.0,234.0,204.0,96.0,2251.0,Lodgepole Pine
1,2724.0,160.0,19.0,60.0,4.0,1350.0,236.0,240.0,127.0,2514.0,Lodgepole Pine
2,2360.0,65.0,7.0,127.0,21.0,1377.0,227.0,226.0,134.0,339.0,Ponderosa Pine
3,2995.0,45.0,4.0,285.0,30.0,5125.0,221.0,231.0,146.0,5706.0,Lodgepole Pine
4,2400.0,106.0,27.0,150.0,63.0,342.0,253.0,196.0,51.0,811.0,Ponderosa Pine
...,...,...,...,...,...,...,...,...,...,...,...
9995,2917.0,90.0,9.0,247.0,25.0,4095.0,235.0,225.0,121.0,3901.0,Lodgepole Pine
9996,3015.0,38.0,8.0,361.0,74.0,4846.0,220.0,223.0,138.0,1611.0,Lodgepole Pine
9997,3052.0,79.0,19.0,90.0,11.0,1003.0,241.0,203.0,85.0,1490.0,Spruce/Fir
9998,2958.0,58.0,6.0,319.0,19.0,2468.0,225.0,227.0,137.0,2280.0,Lodgepole Pine


Separe a coluna `Cover_Type` em uma variável `y` (já que esta é a classe do dataset), e utilize o método `value_counts` para relembrar quantas classes esse problema contém, e qual sua frequência.

In [7]:
y = base['Cover_Type'].values
y.value_counts()

Lodgepole Pine       4847
Ponderosa Pine        581
Spruce/Fir           3714
Aspen                 163
Douglas-fir           278
Krummholz             362
Cottonwood/Willow      55
Name: count, dtype: int64

Agora separe o restante das colunas em uma variável `X`, e já recupere o atributo `values` para ter um NumPy array. Exiba este array.

In [8]:
X = base.drop(columns=['Cover_Type']).values
X

array([[2767.,   66.,   17., ...,  204.,   96., 2251.],
       [2724.,  160.,   19., ...,  240.,  127., 2514.],
       [2360.,   65.,    7., ...,  226.,  134.,  339.],
       ...,
       [3052.,   79.,   19., ...,  203.,   85., 1490.],
       [2958.,   58.,    6., ...,  227.,  137., 2280.],
       [2682.,   91.,   13., ...,  219.,  108., 1661.]], shape=(10000, 10))

Como os dados estão em escalas diferentes, instancie um escalonador do tipo `StandardScaler` e escalone `X` com ele.

In [9]:
scaler = StandardScaler()
X = scaler.fit_transform(X)
X

array([[-0.71700375, -0.78934465,  0.38187624, ..., -0.96994893,
        -1.19597298,  0.19948581],
       [-0.87120088,  0.05516249,  0.6497092 , ...,  0.84361411,
        -0.38987904,  0.39804796],
       [-2.17649753, -0.79832877, -0.95728858, ...,  0.1383396 ,
        -0.20785782, -1.2440535 ],
       ...,
       [ 0.30500049, -0.67255111,  0.6497092 , ..., -1.02032568,
        -1.48200632, -0.37506096],
       [-0.03208161, -0.8612176 , -1.09120506, ...,  0.18871635,
        -0.12984873,  0.22138049],
       [-1.02181203, -0.56474169, -0.15378969, ..., -0.21429766,
        -0.88393662, -0.24595781]], shape=(10000, 10))

Agora instancie e ajuste um algoritmo do tipo `KMeans`. Utilize `n_clusters` igual ao número de classes do dataset.

In [10]:
km = KMeans(n_clusters=7)
km.fit(X)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",7
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",None
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


Recupere e exiba os centroides.

In [11]:
centroids = km.cluster_centers_
centroids

array([[-0.11595415, -0.20419483, -0.57270006,  0.03428311, -0.25233644,
         0.87491589,  0.24472358,  0.22491383,  0.00683081,  2.35213444],
       [ 0.74296866, -0.04189222, -0.01596324,  1.85537521,  1.91839267,
        -0.11357335,  0.03170508,  0.19654339,  0.0879097 , -0.03986544],
       [-0.28915798, -0.65091962, -0.36509321, -0.32192174, -0.42588609,
        -0.6226252 ,  0.54791363,  0.04488911, -0.36296216, -0.21812844],
       [ 0.54585598, -0.58714443, -0.50307421, -0.22832388, -0.38841544,
         1.14983148,  0.52905407,  0.1596045 , -0.27683687,  0.01030476],
       [-1.01474592,  1.33207409,  1.42885965, -0.23112145,  0.37944456,
        -0.54118111, -2.1238366 , -0.32355633,  1.34047374, -0.51579557],
       [ 0.36002753,  1.1532819 , -0.2823173 , -0.12257085, -0.26295801,
         0.11642786, -0.58999695,  0.68803337,  0.90469182, -0.2939059 ],
       [-0.5256239 , -0.7706092 ,  1.34605204, -0.17405112,  0.23088644,
        -0.50832012,  0.57524038, -1.66322079

Aplique o método `inverse_transform` do escalonador para visualizar os centroides na escala original.

In [12]:
X_original = scaler.inverse_transform(X)

In [20]:
scaler.inverse_transform(centroids)

array([[2934.61098398,  131.13157895,    9.87185355,  277.99427918,
          31.92791762, 3725.37528604,  219.04004577,  227.71853547,
         142.25629291, 5102.23112128],
       [3174.13347237,  149.19708029,   14.02919708,  663.10218978,
         158.23253389, 2179.42440042,  213.3649635 ,  227.15537018,
         145.37434828, 1933.97393118],
       [2886.31070288,   81.4077476 ,   11.4221246 ,  202.66733227,
          21.8298722 , 1383.29113419,  227.11741214,  224.14496805,
         128.03514377, 1697.860623  ],
       [3119.16587997,   88.50640593,   10.39177343,  222.46055293,
          24.01011463, 4155.33041133,  226.61496966,  226.42211733,
         131.34726905, 2000.42548887],
       [2683.97048406,  302.12987013,   24.81818182,  221.86894923,
          68.68831169, 1510.66587957,  155.93860685,  216.83116883,
         193.54427391, 1303.59386068],
       [3067.34505088,  282.22895467,   12.04024052,  244.82423682,
          31.30989824, 2539.13552266,  196.80203515,  236

Recupere e exiba os rótulos atribuídos pelo algoritmo KMeans em uma variável `labels`.

In [ ]:
labels = km.predict(X_original)
labels

array([0, 0, 3, ..., 0, 0, 0], shape=(10000,), dtype=int32)

In [21]:
labels = km.labels_
labels

array([6, 2, 2, ..., 6, 2, 2], shape=(10000,), dtype=int32)

Agora aplique o método do cotovelo para verificar se o número de clusters igual a 7 é o melhor valor para agrupar este dataset. Faça uma pesquisa com 1 até 10 clusters, e plote o gráfico correspondente.

In [14]:
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

In [15]:
fig = px.line(x=range(1, 11), y=wcss)
fig.show()

Qual sua conclusão a partir do gráfico?

Vamos visualizar os dados junto com os rótulos atribuídos. Para isso, precisamos utilizar a técnica PCA.

Instancie um objeto PCA com `n_components=2`, e transforme `X` para seu PCA correspondente.

In [16]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

Agora utilize `X_pca` para plotar os dados, usando `labels` para definir a cor dos pontos. Desta vez, utilize `labels.astype(str)` para que a biblioteca considere que os rótulos são categóricos, e não numéricos.

In [17]:
fig = px.scatter(x=X_pca[:,0], y=X_pca[:, 1], color=labels.astype(str))
fig.show()

In [22]:
fig = px.scatter(x=X_pca[:,0], y=X_pca[:, 1], color=labels.astype(str))
fig.show()

Para comparar, refaça o mesmo gráfico, mas agora utilize `y` (ou seja, os rótulos originais) para definir a cor dos pontos.

In [18]:
fig = px.scatter(x=X_pca[:,0], y=X_pca[:, 1],  color=y)
fig.show()

Observe que a comparação deixa evidente que o agrupamento baseado apenas nos atributos resulta em grupos bem diferentes daqueles discriminados pelas espécies das árvores.

DICA: você pode comparar as classes originais com os rótulos gerados pelo algoritmo utilizando a função `pd.crosstab`:

In [23]:
pd.crosstab(labels, y)

col_0,Aspen,Cottonwood/Willow,Douglas-fir,Krummholz,Lodgepole Pine,Ponderosa Pine,Spruce/Fir
row_0,,,,,,,
0,5,0,0,2,614,0,251
1,11,1,2,92,410,18,425
2,69,20,80,31,1398,164,739
3,2,0,0,96,633,1,754
4,8,9,118,8,340,193,172
5,25,1,17,85,982,33,1020
6,43,24,61,48,470,172,353


Isso indica que nenhum rótulo é característico de apenas uma classe, e vice-versa, o que confirma a visualização do gráfico.